#### ENRICH CUSTOMERS FROM BRONZE TO SILVER

In [0]:
import uuid, traceback
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
CATALOG = 'ORDERS_SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'CUSTOMERS_ENRICHED'
TARGET = f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}"
SOURCE = f"{CATALOG}.BRONZE.CUSTOMERS"
AUDIT_TABLE = f"{CATALOG}.AUDIT.PIPELINE_AUDIT_LOG"

In [0]:
cust_df = spark.table(SOURCE)
cust_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

cust_enriched_df = cust_df.select(
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    upper(trim(col("customer_city"))).alias("customer_city"),
    upper(trim(col("customer_state"))).alias("customer_state"),
    "file_name",
    "file_path",
    current_timestamp().alias('load_timestamp')
).filter('customer_id IS NOT NULL')

display(cust_enriched_df.limit(1))

In [0]:
cust_enriched_df.createOrReplaceTempView('enriched_customers_temp_vw')

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.CUSTOMERS_ENRICHED`

In [0]:
from pyspark.sql.functions import *

def write_audit(run_id, status, start_time, rows=None, error=None):
    end = datetime.now()
    row = (run_id, "CUSTOMERS_ENRICHED_LOAD", "SILVER", SOURCE, TARGET, status, rows,
           start_time, end, (end - start_time).total_seconds(),
           error[:4000] if error else None,
           spark.sql("SELECT current_user()").first()[0])
    (spark.createDataFrame([row], spark.table(AUDIT_TABLE).schema)
          .write.mode("append").format("delta").saveAsTable(AUDIT_TABLE))

run_id, start_time = str(uuid.uuid4()), datetime.now()
status, rows_written, err = "SUCCESS", None, None

try:
  result = spark.sql(f"""
  MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
  USING (
    SELECT *
    FROM (
      SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY load_timestamp DESC
            ) rn
      FROM enriched_customers_temp_vw
    )
    WHERE rn = 1
  ) src
  ON (src.customer_id = tgt.customer_id)
  WHEN MATCHED THEN
    UPDATE SET
      tgt.customer_id = src.customer_id,
      tgt.customer_unique_id = src.customer_unique_id,
      tgt.customer_zip_code_prefix = src.customer_zip_code_prefix,
      tgt.customer_city = src.customer_city,
      tgt.customer_state = src.customer_state
  WHEN NOT MATCHED THEN
    INSERT (
      customer_id,
      customer_unique_id,
      customer_zip_code_prefix,
      customer_city,
      customer_state,
      origin_file_path,
      load_timestamp
    )
    VALUES (
      src.customer_id,
      src.customer_unique_id,
      src.customer_zip_code_prefix,
      src.customer_city,
      src.customer_state,
      src.file_path,
      current_timestamp()
    )
  """)

  m = result.first().asDict()
  inserted = m.get("num_inserted_rows") or 0
  updated  = m.get("num_updated_rows") or 0
  rows_written = int(inserted + updated)
  print(f"inserted={inserted}, updated={updated}")

except Exception:
    status, err = "FAILED", traceback.format_exc()
    raise
finally:
    write_audit(run_id, status, start_time, rows_written, err)   

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.CUSTOMERS_ENRICHED')